# Import Libraries

In [15]:
import pandas as pd
import os
import re
from datetime import datetime
from collections import defaultdict
import glob
import pdfplumber
from docx import Document

# Get List of Excel Files

In [16]:
data_dir = r'C:\Users\ALESSANDRO\Documents\GitHub\false_start\data\in\kate'
all_files = glob.glob(os.path.join(data_dir, '**', '*'), recursive=True)
xlsx_files = [f for f in all_files if f.lower().endswith('.xlsx')]
pdf_files = [f for f in all_files if f.lower().endswith('.pdf')]
docx_files = [f for f in all_files if f.lower().endswith('.docx')]
print(f"Found {len(xlsx_files)} XLSX files, {len(pdf_files)} PDF files, {len(docx_files)} DOCX files")

Found 95 XLSX files, 3 PDF files, 7 DOCX files


# Define Extraction Function

In [17]:
def extract_tables_from_file(file_path):
    try:
        excel_file = pd.ExcelFile(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return []
    
    all_tables = []
    for sheet_name in excel_file.sheet_names:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name)
        except Exception as e:
            print(f"Error reading sheet {sheet_name} in {file_path}: {e}")
            continue
        
        if df.empty or df.shape[1] == 0:
            continue
        
        columns_of_interest = ['rank', 'BIB', 'athlete', 'nat', 'finish time', 'reaction time']
        tables = []
        
        # Extract judge's name once from the entire dataset
        judge_info_row = df[df.iloc[:, 0].astype(str).str.contains('judge:', na=False)]
        judge = None
        if not judge_info_row.empty:
            judge_wind_info = str(judge_info_row.iloc[0, 0])
            if "judge:" in judge_wind_info:
                judge = judge_wind_info.split("judge:")[1].split("wind:")[0].strip().split("(")[0].strip()
        
        start_indices = df[df.iloc[:, 0].astype(str).str.contains('rank', na=False)].index
        end_indices = df[df.iloc[:, 0].astype(str).str.contains('judge', na=False)].index
        
        for start, end in zip(start_indices, end_indices):
            try:
                # Extract metadata
                metadata_row = df.iloc[max(0, start-3):start].reset_index(drop=True)
                if len(metadata_row) == 0:
                    continue
                first_row = str(metadata_row.iloc[0, 0]).replace("Result List", "").strip()
                
                # Extract date
                date_match = re.search(r"\d{2}\.\d{2}\.\d{4}", first_row)
                event_date = datetime.strptime(date_match.group(), "%d.%m.%Y").date() if date_match else None
                
                # Extract time
                time_match = re.search(r"\b\d{1,2}:\d{2}\b", first_row)
                event_time = time_match.group() if time_match else None
                
                # Extract category
                category = first_row
                if date_match:
                    category = category.replace(date_match.group(), "").strip()
                if time_match:
                    category = category.replace(time_match.group(), "").strip()
                
                # Extract track length, hurdles, gender
                track_length = re.search(r"\b\d{2,4}m\b", category)
                track_length = track_length.group(0) if track_length else None
                hurdles = "Hurdles" if "Hurdles" in category else None
                gender = "Men" if "Men" in category else "Women" if "Women" in category else None
                
                if track_length:
                    category = category.replace(track_length, "").strip()
                if hurdles:
                    category = category.replace("Hurdles", "").strip()
                if gender:
                    category = category.replace(gender, "").strip()
                
                # Extract WR and MR
                wr_info = metadata_row.iloc[1].dropna().tolist() if len(metadata_row) > 1 else []
                mr_info = metadata_row.iloc[2].dropna().tolist() if len(metadata_row) > 2 else []
                
                wr_athlete, wr_nat, wr_city, wr_date = (None, None, None, None)
                if len(wr_info) >= 5:
                    wr_athlete, wr_nat, wr_city = wr_info[2:5]
                    wr_date = wr_info[5] if len(wr_info) > 5 else None
                
                mr_athlete, mr_nat, mr_date = (None, None, None)
                if len(mr_info) >= 4:
                    mr_athlete, mr_nat = mr_info[2:4]
                    mr_date = mr_info[4] if len(mr_info) > 4 else None
                
                # Extract table
                table = df.iloc[start+1:end].copy()
                table.columns = df.iloc[start]
                table.columns = table.columns.astype(str).str.strip()
                
                try:
                    table = table[columns_of_interest]
                except KeyError:
                    continue
                
                table = table.dropna(how='all')
                
                # Extract wind
                wind_info = str(df.iloc[end, 0])
                wind = None
                if "wind:" in wind_info:
                    wind = wind_info.split("wind:")[1].strip()
                
                # Add metadata
                table['track_length'] = track_length
                table['hurdles'] = hurdles
                table['gender'] = gender
                table['category'] = category
                table['date'] = event_date
                table['time'] = event_time
                table['judge'] = judge
                table['wind'] = wind
                table['wr_athlete'] = wr_athlete
                table['wr_nat'] = wr_nat
                table['wr_city'] = wr_city
                table['wr_date'] = wr_date
                table['mr_athlete'] = mr_athlete
                table['mr_nat'] = mr_nat
                table['mr_date'] = mr_date
                table['source_file'] = os.path.basename(file_path)
                table['sheet_name'] = sheet_name
                
                # Split athlete names
                if 'athlete' in table.columns:
                    table['athlete_surname'] = table['athlete'].str.split(',').str[0].str.strip().str.lower()
                    table['athlete_name'] = table['athlete'].str.split(',').str[1].str.strip().str.lower()
                    table = table.drop(columns=['athlete'])
                
                tables.append(table)
            except Exception as e:
                print(f"Error extracting table from {file_path} sheet {sheet_name} at start {start}: {e}")
                continue
        
        all_tables.extend(tables)
    
    return all_tables

# Process All Files and Merge Tables

In [19]:
all_tables = []
file_summaries = []

# Process XLSX
for file_path in xlsx_files:
    print(f"Processing XLSX {os.path.basename(file_path)}")
    tables = extract_tables_from_file(file_path)
    all_tables.extend(tables)
    file_summaries.append({
        'file': os.path.basename(file_path),
        'type': 'XLSX',
        'num_tables': len(tables),
        'total_rows': sum(len(t) for t in tables)
    })

# Process PDF
for file_path in pdf_files:
    print(f"Processing PDF {os.path.basename(file_path)}")
    tables = extract_from_pdf(file_path)
    all_tables.extend(tables)
    file_summaries.append({
        'file': os.path.basename(file_path),
        'type': 'PDF',
        'num_tables': len(tables),
        'total_rows': sum(len(t) for t in tables)
    })

# Process DOCX
for file_path in docx_files:
    print(f"Processing DOCX {os.path.basename(file_path)}")
    tables = extract_from_docx(file_path)
    all_tables.extend(tables)
    file_summaries.append({
        'file': os.path.basename(file_path),
        'type': 'DOCX',
        'num_tables': len(tables),
        'total_rows': sum(len(t) for t in tables)
    })

# Merge all tables
if all_tables:
    merged_df = pd.concat(all_tables, ignore_index=True, sort=False)
    print(f"Merged dataset shape: {merged_df.shape}")
else:
    print("No tables extracted")
    merged_df = pd.DataFrame()

Processing XLSX 2001 Extract TandF weltklasse Zurich with RT W NAT.xlsx
Processing XLSX 2001_zurich.xlsx
Processing XLSX 2002 2003 TandF EXTRACT Eur Championships with RT IV.xlsx
Processing XLSX 2002 and 2001 TandF EXTRACTS weltklasse Zurich incl RT.xlsx
Processing XLSX 2002 EXTRACT TandF Eur Champsionships II with RT DOB CNTRY .xlsx
Processing XLSX 2002 EXTRACT TandF Eur Champsionships with RT DOB CNTRY.xlsx
Processing XLSX 2002 Extract TandF European CHampionship with RT DOB CNTRY .xlsx
Processing XLSX 2002 Extract TandF Zurich Weltklasse with RT W Nat.xlsx
Processing XLSX 2002 TandF EXTRACT Eur CHampionships with RT III.xlsx
Processing XLSX 2003 2002 2001 2000 TandF EXTRACT zurich with RT.xlsx
Processing XLSX 2003 and 2002 TandF EXTRACT IIAF grand prix II and various.xlsx
Processing XLSX 2003_finland.xlsx
Processing XLSX 1983 to 2003 TandF results for championships no RT.xlsx
Processing XLSX 1983 to 2003 TandF stats only 100 200 400m.xlsx
Processing XLSX 1998 TandF IIAF Zurich resul

# Save Merged Dataset

In [20]:
output_path = r'C:\Users\ALESSANDRO\Documents\GitHub\false_start\data\merged_kate_data.xlsx'
merged_df.to_excel(output_path, index=False)
print(f"Merged data saved to {output_path}")

Merged data saved to C:\Users\ALESSANDRO\Documents\GitHub\false_start\data\merged_kate_data.xlsx


# Data Summary

In [21]:
print("Summary of Data:")
print(f"Total files processed: {len(file_summaries)}")
print(f"Total tables extracted: {sum(s['num_tables'] for s in file_summaries)}")
print(f"Total rows in merged dataset: {len(merged_df)}")
print("\nFiles with data:")
for s in file_summaries:
    if s['num_tables'] > 0:
        print(f"- {s['file']}: {s['num_tables']} tables, {s['total_rows']} rows")

print("\nColumns in merged dataset:")
print(list(merged_df.columns))

print("\nSample data:")
display(merged_df.head())

Summary of Data:
Total files processed: 105
Total tables extracted: 118
Total rows in merged dataset: 801

Files with data:
- 2001 Extract TandF weltklasse Zurich with RT W NAT.xlsx: 11 tables, 96 rows
- 2001_zurich.xlsx: 11 tables, 96 rows
- 2002 and 2001 TandF EXTRACTS weltklasse Zurich incl RT.xlsx: 13 tables, 0 rows
- 2002 Extract TandF Zurich Weltklasse with RT W Nat.xlsx: 8 tables, 64 rows
- 2000 TandF golden league and schedule for 1998 event without results.xlsx: 9 tables, 71 rows
- 2001 Extract TandF weltklasse Zurich with RT W NAT.xlsx: 11 tables, 96 rows
- 2002 and 2001 TandF EXTRACTS weltklasse Zurich incl RT.xlsx: 13 tables, 0 rows
- 2002 and 2001 TandF weltklasse Zurich incl RT.xlsx: 20 tables, 168 rows
- 2002 Extract TandF Zurich Weltklasse with RT W Nat.xlsx: 8 tables, 64 rows
- 2003 TandF Bislett games with RT.xlsx: 5 tables, 89 rows
- 2003 TandF EXTRACT Bislett games with RT.xlsx: 3 tables, 24 rows
- 2004 2003 TandF Bislett games Bergen.xlsx: 4 tables, 28 rows
- 1983 

,rank,BIB,nat,finish time,reaction time,track_length,hurdles,gender,category,date,...,mr_athlete,mr_nat,mr_date,source_file,sheet_name,athlete_surname,athlete_name,www.iaaf.org\nstatistics handbook,"6-8, quai antoine 1er | bp 359 | mc 98007 | monaco cedex\ntel: +377 93 10 88 88 fax: +377 93 15 95 15",NaN
0,1,213,RUS,48.58,0.262,400m,Hurdles,Men,B-Series,2001-08-17,...,Samuel Matete,ZAM,1991-08-07 00:00:00,2001 Extract TandF weltklasse Zurich with RT W...,Sheet1,gorban,boris,NaN,NaN,NaN
1,2,218,POL,48.69,0.199,400m,Hurdles,Men,B-Series,2001-08-17,...,Samuel Matete,ZAM,1991-08-07 00:00:00,2001 Extract TandF weltklasse Zurich with RT W...,Sheet1,plawgo,marek,NaN,NaN,NaN
2,3,467,JAM,49.08,0.29,400m,Hurdles,Men,B-Series,2001-08-17,...,Samuel Matete,ZAM,1991-08-07 00:00:00,2001 Extract TandF weltklasse Zurich with RT W...,Sheet1,weakly,ian,NaN,NaN,NaN
3,4,217,CZE,49.09,0.312,400m,Hurdles,Men,B-Series,2001-08-17,...,Samuel Matete,ZAM,1991-08-07 00:00:00,2001 Extract TandF weltklasse Zurich with RT W...,Sheet1,muzik,jiri,NaN,NaN,NaN
4,5,215,RUS,49.13,0.262,400m,Hurdles,Men,B-Series,2001-08-17,...,Samuel Matete,ZAM,1991-08-07 00:00:00,2001 Extract TandF weltklasse Zurich with RT W...,Sheet1,machtchenko,rouslan,NaN,NaN,NaN


# Functions for PDF and DOCX

In [18]:
def extract_from_pdf(file_path):
    tables = []
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_tables = page.extract_tables()
                for table in page_tables:
                    if len(table) > 1:
                        df = pd.DataFrame(table[1:], columns=table[0])
                        df.columns = df.columns.astype(str).str.strip().str.lower()
                        # Try to standardize
                        df = df.rename(columns={
                            'rank': 'rank',
                            'bib': 'BIB',
                            'athlete': 'athlete',
                            'nat': 'nat',
                            'finish time': 'finish time',
                            'reaction time': 'reaction time'
                        })
                        if 'athlete' in df.columns:
                            df['athlete_surname'] = df['athlete'].str.split(',').str[0].str.strip().str.lower()
                            df['athlete_name'] = df['athlete'].str.split(',').str[1].str.strip().str.lower()
                            df = df.drop(columns=['athlete'])
                        df['source_file'] = os.path.basename(file_path)
                        df['sheet_name'] = 'PDF_table'
                        tables.append(df)
    except Exception as e:
        print(f"Error processing PDF {file_path}: {e}")
    return tables

def extract_from_docx(file_path):
    tables = []
    try:
        doc = Document(file_path)
        for i, table in enumerate(doc.tables):
            data = [[cell.text.strip() for cell in row.cells] for row in table.rows]
            if len(data) > 1:
                df = pd.DataFrame(data[1:], columns=data[0])
                df.columns = df.columns.astype(str).str.strip().str.lower()
                # Standardize
                df = df.rename(columns={
                    'rank': 'rank',
                    'bib': 'BIB',
                    'athlete': 'athlete',
                    'nat': 'nat',
                    'finish time': 'finish time',
                    'reaction time': 'reaction time'
                })
                if 'athlete' in df.columns:
                    df['athlete_surname'] = df['athlete'].str.split(',').str[0].str.strip().str.lower()
                    df['athlete_name'] = df['athlete'].str.split(',').str[1].str.strip().str.lower()
                    df = df.drop(columns=['athlete'])
                df['source_file'] = os.path.basename(file_path)
                df['sheet_name'] = f'DOCX_table_{i}'
                tables.append(df)
    except Exception as e:
        print(f"Error processing DOCX {file_path}: {e}")
    return tables

In [22]:
# Build inclusion status list for all files
included_files = {s['file']: s for s in file_summaries}
records = []
for path in sorted(all_files):
    file_name = os.path.basename(path)
    if file_name == '':
        continue
    summary = included_files.get(file_name)
    included = 'YES' if summary and summary['total_rows'] > 0 else 'NO'
    records.append({
        'file': file_name,
        'path': path,
        'type': 'XLSX' if file_name.lower().endswith('.xlsx') else 'PDF' if file_name.lower().endswith('.pdf') else 'DOCX' if file_name.lower().endswith('.docx') else 'OTHER',
        'included': included,
        'tables_extracted': summary['num_tables'] if summary else 0,
        'rows_extracted': summary['total_rows'] if summary else 0
    })

inclusion_df = pd.DataFrame(records)
output_list_path = os.path.join(r'C:\Users\ALESSANDRO\Documents\GitHub\false_start\data', 'kate_file_inclusion_list.csv')
inclusion_df.to_csv(output_list_path, index=False)
print(f"Saved file inclusion list to: {output_list_path}")
print(inclusion_df[['file','type','included','tables_extracted','rows_extracted']].to_string(index=False))

Saved file inclusion list to: C:\Users\ALESSANDRO\Documents\GitHub\false_start\data\kate_file_inclusion_list.csv
                                                                               file  type included  tables_extracted  rows_extracted
                                                                              1.zip OTHER       NO                 0               0
                                                                             10.zip OTHER       NO                 0               0
                                                                             11.zip OTHER       NO                 0               0
                                                                             12.zip OTHER       NO                 0               0
                                                                             13.zip OTHER       NO                 0               0
                                                                             14.zip OTHER

In [24]:
preview = pd.read_csv(output_list_path)
print(f"Total files listed: {len(preview)}")
print(preview['included'].value_counts())
print(preview[['file','type','included','tables_extracted','rows_extracted']].head(10).to_string(index=False))

Total files listed: 120
included
NO     109
YES     11
Name: count, dtype: int64
                                                   file  type included  tables_extracted  rows_extracted
                                                  1.zip OTHER       NO                 0               0
                                                 10.zip OTHER       NO                 0               0
                                                 11.zip OTHER       NO                 0               0
                                                 12.zip OTHER       NO                 0               0
                                                 13.zip OTHER       NO                 0               0
                                                 14.zip OTHER       NO                 0               0
                                                 15.zip OTHER       NO                 0               0
                                                  2.zip OTHER       NO         